# A report in Jupyter

Before opening this notebook, install the dependencies in the kernel environment:

```sh
pip install 'report-toolkit[notebook,pandas,matplotlib,altair]'
```

From a project checkout, use `uv sync --extra notebook --extra pandas --extra matplotlib --extra altair`.

Select the **Python (report-toolkit)** kernel. This example combines a table, Matplotlib and Altair charts, text, lists, and HTML in columns. The Altair charts appear side by side with tooltips. Objects use native notebook display.

In [ ]:
%matplotlib inline

import altair as alt
import matplotlib.pyplot as plt
import pandas as pd

from report_toolkit import Report

sales = pd.DataFrame(
    {
        'Month': ['January', 'February', 'March'],
        'Revenue (BRL)': [12000, 15000, 18000],
    }
)

In [ ]:
report = Report('Quarterly sales', description='Illustrative data')
report.markdown('A quick overview of **monthly sales**.')
report.heading(1, 'Overview')

with report.columns(2):
    with report.panel('Data'):
        report.add(sales)
    with report.panel('Summary'):
        total = sales['Revenue (BRL)'].sum()
        report.markdown(f'Total revenue: **BRL {total:,.0f}**')
        report.unordered(
            [
                'Revenue increased every month.',
                'March had the best result.',
            ]
        )

In [ ]:
fig, ax = plt.subplots(figsize=(4, 2.5))
ax.plot(sales['Month'], sales['Revenue (BRL)'], marker='o')
ax.set(ylabel='Revenue (BRL)', title='Monthly trend')
ax.grid(axis='y', alpha=0.3)
fig.tight_layout()
plt.close(fig)  # Display the figure only inside the report.

with report.section('Trend and observations'), report.columns(2):
    with report.panel('Chart'):
        report.add(fig)
    with report.panel('Interpreting the results'):
        growth = sales['Revenue (BRL)'].iloc[-1] / sales['Revenue (BRL)'].iloc[0] - 1
        report.markdown(f'Growth over the period: **{growth:.0%}**.')
        report.markdown(
            '> These values are illustrative and do not represent a forecast.'
        )
        report.raw_html('<p><strong>Source:</strong> demonstration data.</p>')

In [ ]:
months = sales['Month'].tolist()
base = (
    alt.Chart(sales)
    .encode(
        x=alt.X('Month:N', sort=months, axis=alt.Axis(labelAngle=0)),
        y=alt.Y('Revenue (BRL):Q', title='Revenue (BRL)'),
        tooltip=['Month:N', alt.Tooltip('Revenue (BRL):Q', format=',.0f')],
    )
    .properties(width=200, height=160)
)

bars = base.mark_bar(color='#4c78a8').properties(title='Monthly revenue')
line = base.mark_line(point=True, color='#f58518').properties(title='Revenue trend')

cumulative = sales.assign(Cumulative=sales['Revenue (BRL)'].cumsum())
area = (
    alt.Chart(cumulative)
    .mark_area(line=True, point=True, color='#54a24b', opacity=0.4)
    .encode(
        x=alt.X('Month:N', sort=months, axis=alt.Axis(labelAngle=0)),
        y=alt.Y('Cumulative:Q', title='Cumulative revenue (BRL)'),
        tooltip=['Month:N', alt.Tooltip('Cumulative:Q', format=',.0f')],
    )
    .properties(width=200, height=160, title='Cumulative revenue')
)

report.heading(1, 'Altair charts')
report.markdown('Hover over the charts to inspect the values.')
with report.columns(3):
    report.add(bars)
    report.add(line)
    report.add(area)

In [ ]:
report.heading(1, 'Next steps')
report.ordered(
    [
        'Validate the data.',
        ['Check the values for each month.', 'Check for missing sales.'],
        'Compare the result with the next quarter.',
    ]
)
report.markdown(
    'The total was calculated with:\n\n```python\nsales["Revenue (BRL)"].sum()\n```'
)

In [ ]:
report.display()